In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # ensures reproducibility — same "random" data every run

n = 20000

# customer_id: simulate repeat customers, not all-unique
customer_id = np.random.randint(1000, 5000, size=n)  # ~4000 unique customers across 20000 orders → repeat buyers

print(customer_id[:10])
print("Unique customers:", len(np.unique(customer_id)))

[4174 4507 1860 2294 2130 2095 4772 4092 2638 3169]
Unique customers: 3965


In [2]:
categories = ['apparel', 'electronics', 'groceries', 'books', 'home_furniture']
category_probs = [0.35, 0.15, 0.25, 0.15, 0.10]  # sums to 1.0 — must, since np.random.choice requires it

product_category = np.random.choice(categories, size=n, p=category_probs)

print(pd.Series(product_category).value_counts())

apparel           7032
groceries         5052
books             2964
electronics       2949
home_furniture    2003
Name: count, dtype: int64


In [3]:
# price ranges per category (min, max) for a uniform-ish draw, will make it a bit realistic using lognormal-like skew
price_ranges = {
    'apparel': (300, 3000),
    'electronics': (1000, 50000),
    'groceries': (50, 1500),
    'books': (100, 1200),
    'home_furniture': (500, 20000)
}

item_price = np.array([
    np.random.uniform(*price_ranges[cat]) for cat in product_category
])

quantity = np.random.randint(1, 5, size=n)  # 1 to 4 units per order

order_value = item_price * quantity

print(pd.DataFrame({'category': product_category, 'price': item_price, 'qty': quantity, 'order_value': order_value}).groupby('category')['price'].describe())

                 count          mean           std          min           25%  \
category                                                                        
apparel         7032.0   1654.644589    783.585610   300.276406    984.070943   
books           2964.0    655.905536    314.685070   100.272826    385.284516   
electronics     2949.0  25036.641987  14159.339317  1004.570133  12638.526965   
groceries       5052.0    769.547935    416.305739    50.294590    411.227555   
home_furniture  2003.0  10383.069679   5616.158355   527.346173   5638.576790   

                         50%           75%           max  
category                                                  
apparel          1659.843530   2332.685153   2999.729148  
books             659.885656    924.789882   1199.371369  
electronics     24583.064839  37540.811709  49990.615578  
groceries         770.082507   1126.425212   1499.307519  
home_furniture  10147.739324  15402.012353  19996.274982  


In [4]:
payment_methods = ['credit_card', 'debit_card', 'upi', 'netbanking', 'cod']
payment_method_probs = [0.25, 0.20, 0.35, 0.10, 0.10]  # UPI dominant, reflecting real Indian e-commerce patterns

payment_method = np.random.choice(payment_methods, size=n, p=payment_method_probs)

# payment_attempts: most orders succeed on first try; COD has no real "attempt" concept, so force it to 1
payment_attempts = np.where(
    payment_method == 'cod',
    1,
    np.random.choice([1, 2, 3], size=n, p=[0.75, 0.20, 0.05])
)

print(pd.Series(payment_method).value_counts())
print(pd.DataFrame({'method': payment_method, 'attempts': payment_attempts}).groupby('method')['attempts'].value_counts())

upi            6844
credit_card    5007
debit_card     4030
cod            2070
netbanking     2049
Name: count, dtype: int64
method       attempts
cod          1           2070
credit_card  1           3827
             2            940
             3            240
debit_card   1           2996
             2            810
             3            224
netbanking   1           1543
             2            389
             3            117
upi          1           5147
             2           1375
             3            322
Name: count, dtype: int64


In [5]:
# order_date: random dates across a ~1 year window
order_date = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n), unit='D')

# delivery_days: most orders 2-7 days, some outliers longer (matches real logistics variance)
delivery_days = np.random.choice(
    [2, 3, 4, 5, 6, 7, 10, 14],
    size=n,
    p=[0.15, 0.20, 0.20, 0.15, 0.10, 0.10, 0.06, 0.04]
)
delivery_date = order_date + pd.to_timedelta(delivery_days, unit='D')

# billing/shipping country: mostly India, small fraction mismatched (gifting/dropship scenario)
countries = ['India', 'USA', 'UK', 'UAE', 'Singapore']
billing_country = np.random.choice(countries, size=n, p=[0.90, 0.03, 0.03, 0.02, 0.02])

# shipping_country: mostly same as billing, small chance of mismatch
mismatch_flag_raw = np.random.rand(n) < 0.05  # 5% of orders have a genuinely different shipping country
shipping_country = np.where(
    mismatch_flag_raw,
    np.random.choice(countries, size=n),
    billing_country
)

address_mismatch = (billing_country != shipping_country).astype(int)

print("Delivery days distribution:\n", pd.Series(delivery_days).value_counts().sort_index())
print("\nAddress mismatch rate:", address_mismatch.mean())
print("\nBilling country distribution:\n", pd.Series(billing_country).value_counts())


Delivery days distribution:
 2     3014
3     3991
4     4029
5     2990
6     1937
7     2072
10    1147
14     820
Name: count, dtype: int64

Address mismatch rate: 0.03865

Billing country distribution:
 India        17926
USA            626
UK             611
Singapore      425
UAE            412
Name: count, dtype: int64


In [6]:
# assign each unique customer a hidden return-propensity (not a visible column)
unique_customers = np.unique(customer_id)
customer_return_propensity = pd.Series(
    np.random.beta(a=2, b=8, size=len(unique_customers)),  # skewed toward low propensity, some high-propensity customers
    index=unique_customers
)

# map each row's customer to their propensity
row_propensity = customer_id_series = pd.Series(customer_id).map(customer_return_propensity)

# past_returns: count drawn from a Poisson-like process scaled by propensity (higher propensity -> more past returns)
customer_past_returns = np.random.poisson(lam=row_propensity * 8)

# past_chargebacks: much rarer, independent-ish ceiling, scaled down further
customer_past_chargebacks = np.random.poisson(lam=row_propensity * 0.5)

print(pd.Series(customer_past_returns).describe())
print(pd.Series(customer_past_chargebacks).describe())
print(pd.Series(customer_past_returns).value_counts().sort_index().head(10))

count    20000.000000
mean         1.613950
std          1.608833
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         12.000000
dtype: float64
count    20000.000000
mean         0.101050
std          0.325491
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.000000
dtype: float64
0    5805
1    5503
2    3891
3    2362
4    1222
5     673
6     309
7     136
8      64
9      21
Name: count, dtype: int64


In [7]:
# assemble everything generated so far into one DataFrame
df = pd.DataFrame({
    'customer_id': customer_id,
    'customer_past_returns': customer_past_returns,
    'customer_past_chargebacks': customer_past_chargebacks,
    'product_category': product_category,
    'item_price': item_price,
    'quantity': quantity,
    'order_value': order_value,
    'payment_method': payment_method,
    'payment_attempts': payment_attempts,
    'order_date': order_date,
    'delivery_days': delivery_days,
    'delivery_date': delivery_date,
    'billing_country': billing_country,
    'shipping_country': shipping_country,
    'address_mismatch': address_mismatch,
})

# --- is_returned probability formula (Step 3/4 design, now with real values) ---

max_price = df['item_price'].max()

category_bump = df['product_category'].map({
    'apparel': 0.7, 'electronics': 0.3, 'groceries': 0.0, 'books': 0.0, 'home_furniture': 0.2
})

log_odds_return = (
    -1.8
    + 0.35 * np.minimum(df['customer_past_returns'], 5)
    + 0.9 * (df['item_price'] / max_price)
    + 0.6 * (df['delivery_days'] / 15)
    + 0.25 * (df['payment_attempts'] > 1).astype(int)
    + 0.4 * df['address_mismatch']
    + category_bump
)

return_probability = 1 / (1 + np.exp(-log_odds_return))

# sample the actual label from the probability (this is what injects realistic randomness, not determinism)
df['is_returned'] = (np.random.rand(n) < return_probability).astype(int)

print("Overall is_returned rate:", df['is_returned'].mean())
print(df['is_returned'].value_counts())
print("\nReturn probability stats:\n", return_probability.describe())

Overall is_returned rate: 0.37345
is_returned
0    12531
1     7469
Name: count, dtype: int64

Return probability stats:
 count    20000.000000
mean         0.370726
std          0.144384
min          0.152019
25%          0.267312
50%          0.350657
75%          0.456838
max          0.859216
dtype: float64


In [8]:
log_odds_return = (
    -2.7
    + 0.35 * np.minimum(df['customer_past_returns'], 5)
    + 0.9 * (df['item_price'] / max_price)
    + 0.6 * (df['delivery_days'] / 15)
    + 0.25 * (df['payment_attempts'] > 1).astype(int)
    + 0.4 * df['address_mismatch']
    + category_bump
)

return_probability = 1 / (1 + np.exp(-log_odds_return))
df['is_returned'] = (np.random.rand(n) < return_probability).astype(int)

print("Overall is_returned rate:", df['is_returned'].mean())
print(df['is_returned'].value_counts())

Overall is_returned rate: 0.20395
is_returned
0    15921
1     4079
Name: count, dtype: int64


In [9]:
# does return rate increase with past_returns, as intended?
print(df.groupby('customer_past_returns')['is_returned'].mean())

print()
# does return rate vary by category as intended (apparel highest, groceries/books near zero bump)?
print(df.groupby('product_category')['is_returned'].mean().sort_values(ascending=False))

print()
# does address_mismatch increase return rate?
print(df.groupby('address_mismatch')['is_returned'].mean())

customer_past_returns
0     0.121964
1     0.168090
2     0.215883
3     0.284505
4     0.360884
5     0.423477
6     0.398058
7     0.360294
8     0.359375
9     0.333333
10    0.444444
11    0.500000
12    0.333333
Name: is_returned, dtype: float64

product_category
electronics       0.255341
apparel           0.254408
home_furniture    0.180729
groceries         0.152415
books             0.136640
Name: is_returned, dtype: float64

address_mismatch
0    0.199771
1    0.307891
Name: is_returned, dtype: float64


In [10]:
category_bump = df['product_category'].map({
    'apparel': 1.1, 'electronics': -0.2, 'groceries': -0.3, 'books': -0.3, 'home_furniture': 0.0
})

log_odds_return = (
    -2.7
    + 0.35 * np.minimum(df['customer_past_returns'], 5)
    + 0.9 * (df['item_price'] / max_price)
    + 0.6 * (df['delivery_days'] / 15)
    + 0.25 * (df['payment_attempts'] > 1).astype(int)
    + 0.4 * df['address_mismatch']
    + category_bump
)

return_probability = 1 / (1 + np.exp(-log_odds_return))
df['is_returned'] = (np.random.rand(n) < return_probability).astype(int)

print("Overall is_returned rate:", df['is_returned'].mean())
print(df.groupby('product_category')['is_returned'].mean().sort_values(ascending=False))

Overall is_returned rate: 0.21195
product_category
apparel           0.341297
electronics       0.194981
home_furniture    0.175736
books             0.116397
groceries         0.112233
Name: is_returned, dtype: float64


In [11]:
# days_to_return: only meaningful where is_returned == 1
# use a skewed distribution - most returns happen quickly, some right at the window's edge
days_to_return_raw = np.random.exponential(scale=4, size=n).astype(int) + 1  # skewed: many quick returns, fewer late ones
days_to_return_raw = np.clip(days_to_return_raw, 1, 30)  # cap at a realistic 30-day return window

df['days_to_return'] = np.where(df['is_returned'] == 1, days_to_return_raw, np.nan)
df['return_date'] = np.where(
    df['is_returned'] == 1,
    df['delivery_date'] + pd.to_timedelta(days_to_return_raw, unit='D'),
    pd.NaT
)

print(df[df['is_returned'] == 1][['delivery_date', 'days_to_return', 'return_date']].head())
print("\ndays_to_return stats (returned orders only):\n", df[df['is_returned']==1]['days_to_return'].describe())

   delivery_date  days_to_return          return_date
3     2025-07-30            16.0  1755216000000000000
6     2025-02-12             2.0  1739491200000000000
7     2025-03-27             2.0  1743206400000000000
14    2025-07-02             1.0  1751500800000000000
23    2025-07-22             5.0  1753574400000000000

days_to_return stats (returned orders only):
 count    4239.000000
mean        4.447511
std         3.926413
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        26.000000
Name: days_to_return, dtype: float64


In [12]:
return_date_computed = df['delivery_date'] + pd.to_timedelta(days_to_return_raw, unit='D')

df['return_date'] = return_date_computed.where(df['is_returned'] == 1, pd.NaT)

print(df[df['is_returned'] == 1][['delivery_date', 'days_to_return', 'return_date']].head())
print(df['return_date'].dtype)

   delivery_date  days_to_return return_date
3     2025-07-30            16.0  2025-08-15
6     2025-02-12             2.0  2025-02-14
7     2025-03-27             2.0  2025-03-29
14    2025-07-02             1.0  2025-07-03
23    2025-07-22             5.0  2025-07-27
datetime64[ns]


In [13]:
max_order_value = df['order_value'].max()

payment_method_bump = df['payment_method'].map({
    'credit_card': 0.4, 'debit_card': 0.3, 'upi': 0.1, 'netbanking': 0.1, 'cod': -3.0  # COD: near-zero chargeback risk
})

log_odds_chargeback = (
    -4.3  # low base rate, since chargebacks are rare (~5% or lower target)
    + 0.9 * df['address_mismatch']
    + 0.5 * (df['payment_attempts'] > 1).astype(int)
    + 0.8 * np.minimum(df['customer_past_chargebacks'], 3)
    + 0.7 * (df['order_value'] / max_order_value)
    + payment_method_bump
)

chargeback_probability = 1 / (1 + np.exp(-log_odds_chargeback))
df['is_chargeback'] = (np.random.rand(n) < chargeback_probability).astype(int)

print("Overall is_chargeback rate:", df['is_chargeback'].mean())
print(df.groupby('payment_method')['is_chargeback'].mean())
print(df.groupby('address_mismatch')['is_chargeback'].mean())

Overall is_chargeback rate: 0.02175
payment_method
cod            0.000000
credit_card    0.027162
debit_card     0.027047
netbanking     0.020498
upi            0.021625
Name: is_chargeback, dtype: float64
address_mismatch
0    0.020856
1    0.043984
Name: is_chargeback, dtype: float64


In [14]:
df.to_csv('../data/return_risk_chargeback_dataset.csv', index=False)
print(df.shape)
print(df.dtypes)

(20000, 19)
customer_id                           int32
customer_past_returns                 int32
customer_past_chargebacks             int32
product_category                     object
item_price                          float64
quantity                              int32
order_value                         float64
payment_method                       object
payment_attempts                      int64
order_date                   datetime64[ns]
delivery_days                         int64
delivery_date                datetime64[ns]
billing_country                      object
shipping_country                     object
address_mismatch                      int64
is_returned                           int64
days_to_return                      float64
return_date                  datetime64[ns]
is_chargeback                         int64
dtype: object


In [15]:
df.head()

,customer_id,customer_past_returns,customer_past_chargebacks,product_category,item_price,quantity,order_value,payment_method,payment_attempts,order_date,delivery_days,delivery_date,billing_country,shipping_country,address_mismatch,is_returned,days_to_return,return_date,is_chargeback
0,4174,1,0,groceries,766.884816,3,2300.654448,upi,1,2025-10-11,10,2025-10-21,UAE,UAE,0,0,NaN,NaT,0
1,4507,0,0,electronics,13014.854628,1,13014.854628,credit_card,3,2025-02-26,4,2025-03-02,India,India,0,0,NaN,NaT,0
2,1860,1,0,home_furniture,2078.900125,3,6236.700375,upi,1,2025-08-13,5,2025-08-18,India,India,0,0,NaN,NaT,0
3,2294,1,0,apparel,987.297130,1,987.297130,cod,1,2025-07-28,2,2025-07-30,India,India,0,1,16.0,2025-08-15,0
4,2130,1,0,apparel,465.180864,1,465.180864,upi,3,2025-05-21,6,2025-05-27,UK,UK,0,0,NaN,NaT,1
